<a href="https://colab.research.google.com/github/Anishamalar/AI-robot-people-detector/blob/main/AI_Room_Robot_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics gradio opencv-python-headless

In [ ]:
import cv2, math, os, subprocess, tempfile
import numpy as np
import gradio as gr
from collections import Counter, defaultdict, deque
from ultralytics import YOLO

# YOLOv8 pose model: detects people + 17 body keypoints (auto-downloads on first run)
model = YOLO("yolov8m-pose.pt")

KP_CONF = 0.4
SKELETON = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
            (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]


def _mid(points):
    points = [p for p in points if p is not None]
    return np.mean(points, axis=0) if points else None


def analyze_person(k, box):
    """Rule-based activity recognition from body keypoints.
    Returns posture, arm_action, body_center, raised_wrist_x, scale."""
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    scale = max(w, h, 1.0)

    def pt(i):
        return k[i][:2] if k[i][2] > KP_CONF else None

    S = _mid([pt(5), pt(6)])        # shoulders
    Hp = _mid([pt(11), pt(12)])     # hips
    K = _mid([pt(13), pt(14)])      # knees
    A = _mid([pt(15), pt(16)])      # ankles
    nose = pt(0)

    center = Hp if Hp is not None else np.array([(x1 + x2) / 2, (y1 + y2) / 2])
    tl = np.linalg.norm(Hp - S) if (S is not None and Hp is not None) else None

    # ---- posture ----
    posture = "standing"
    if w > 1.2 * h:
        posture = "lying down"
    elif tl is not None:
        d = Hp - S
        angle = math.degrees(math.atan2(abs(d[0]), abs(d[1]) + 1e-6))
        if angle > 60:
            posture = "lying down"
        elif K is not None and abs(K[1] - Hp[1]) < 0.6 * tl:
            posture = "sitting"
        elif K is None and A is None:
            posture = "upper body visible (sitting/at desk)"
    elif K is None and A is None:
        posture = "partially visible"

    # ---- arms ----
    arm, raised_x = None, None
    if S is not None:
        ref = tl if tl else 0.3 * h
        raised = []
        for wi in (9, 10):
            wr = pt(wi)
            if wr is not None and wr[1] < S[1] - 0.2 * ref:
                raised.append(wr)
        if raised:
            arm = "hands raised"
            raised_x = float(np.mean([p[0] for p in raised]))
        elif nose is not None:
            for wi in (9, 10):
                wr = pt(wi)
                if wr is not None and np.linalg.norm(wr - nose) < 0.5 * ref:
                    arm = "hand near face (phone/drinking)"
                    break
    return posture, arm, center, raised_x, scale


def smooth(buf, value):
    buf.append(value)
    return Counter(buf).most_common(1)[0][0]


def draw_robot(img, x, y, s=34):
    cv2.rectangle(img, (x, y), (x + s, y + s), (255, 200, 0), -1)
    cv2.circle(img, (x + s // 4, y + s // 2 - 3), 4, (0, 0, 0), -1)
    cv2.circle(img, (x + 3 * s // 4, y + s // 2 - 3), 4, (0, 0, 0), -1)
    cv2.rectangle(img, (x + s // 4, y + 3 * s // 4 - 2), (x + 3 * s // 4, y + 3 * s // 4 + 2), (0, 0, 0), -1)
    cv2.line(img, (x + s // 2, y), (x + s // 2, y - 8), (255, 200, 0), 2)
    cv2.circle(img, (x + s // 2, y - 10), 3, (0, 0, 255), -1)


def put_text(img, text, org, scale=0.55, color=(255, 255, 255), bg=(0, 0, 0)):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    x, y = org
    cv2.rectangle(img, (x - 2, y - th - 3), (x + tw + 2, y + bl), bg, -1)
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)


def fmt_t(sec):
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"


def analyze(video_path, progress=gr.Progress()):
    if video_path is None:
        return None, "⚠️ Please upload a video first."

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1

    raw_path = tempfile.mktemp(suffix=".mp4")
    out_path = tempfile.mktemp(suffix=".mp4")
    writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    model.predictor = None  # reset tracker for a new video
    hist = {}                                   # id -> recent (cx, cy, scale, wrist_x)
    prim_buf, arm_buf = {}, {}                  # smoothing buffers
    prim_time, arm_time = defaultdict(Counter), defaultdict(Counter)
    seen = {}                                   # id -> [first_t, last_t]
    counts, log = [], []
    sec_step = max(1, round(fps))
    fi = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = fi / fps
        r = model.track(frame, persist=True, classes=[0], conf=0.35,
                        imgsz=640, verbose=False)[0]

        people = []
        if r.boxes is not None and r.boxes.id is not None and r.keypoints is not None:
            ids = r.boxes.id.int().cpu().tolist()
            boxes = r.boxes.xyxy.cpu().numpy()
            kps = r.keypoints.data.cpu().numpy()

            for pid, b, k in zip(ids, boxes, kps):
                posture, arm, center, rx, scale = analyze_person(k, b)
                hh = hist.setdefault(pid, deque(maxlen=max(3, int(fps * 1.5))))
                hh.append((center[0], center[1], scale, rx))

                # ---- movement (walking / running) ----
                motion = None
                if posture in ("standing", "partially visible") and len(hh) >= max(3, int(fps * 0.5)):
                    p0, p1 = hh[0], hh[-1]
                    dt = (len(hh) - 1) / fps
                    sp = math.hypot(p1[0] - p0[0], p1[1] - p0[1]) / np.mean([q[2] for q in hh]) / dt
                    motion = "running" if sp > 1.6 else "walking" if sp > 0.25 else None

                # ---- waving ----
                if arm == "hands raised":
                    xs = [q[3] for q in hh if q[3] is not None]
                    if len(xs) >= max(3, int(fps * 0.5)) and np.std(xs) > 0.12 * scale:
                        arm = "waving"

                prim = smooth(prim_buf.setdefault(pid, deque(maxlen=9)), motion or posture)
                armv = smooth(arm_buf.setdefault(pid, deque(maxlen=9)), arm or "")
                armv = armv or None

                prim_time[pid][prim] += 1 / fps
                if armv:
                    arm_time[pid][armv] += 1 / fps
                seen.setdefault(pid, [t, t])[1] = t
                people.append((pid, b, k, prim, armv))

        n = len(people)
        counts.append(n)

        # ---- draw ----
        for pid, b, k, prim, armv in people:
            x1, y1, x2, y2 = map(int, b)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 220, 120), 2)
            for a, c in SKELETON:
                if k[a][2] > KP_CONF and k[c][2] > KP_CONF:
                    cv2.line(frame, tuple(map(int, k[a][:2])), tuple(map(int, k[c][:2])), (255, 160, 0), 2)
            label = f"#{pid} {prim}" + (f" + {armv}" if armv else "")
            put_text(frame, label, (x1, max(18, y1 - 6)), 0.5, (0, 0, 0), (0, 220, 120))

        band = 52 + 20 * min(n, 8)
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (min(W, 520), band), (30, 30, 30), -1)
        frame = cv2.addWeighted(overlay, 0.65, frame, 0.35, 0)
        draw_robot(frame, 12, 18)
        cv2.putText(frame, f"AI ROBOT: I see {n} person{'s' if n != 1 else ''}",
                    (60, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        for i, (pid, _, _, prim, armv) in enumerate(people[:8]):
            cv2.putText(frame, f"#{pid}: {prim}" + (f", {armv}" if armv else ""),
                        (60, 66 + 20 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 255, 200), 1, cv2.LINE_AA)

        # ---- 1-second log ----
        if fi % sec_step == 0:
            desc = "; ".join(f"#{p[0]} {p[3]}" + (f" ({p[4]})" if p[4] else "") for p in people) or "nobody"
            log.append(f"`{fmt_t(t)}` – **{n}** people: {desc}")

        writer.write(frame)
        fi += 1
        if fi % 10 == 0:
            progress(fi / total, desc=f"Analyzing frame {fi}/{total}")

    cap.release()
    writer.release()

    # browser-friendly H.264
    subprocess.run(["ffmpeg", "-y", "-i", raw_path, "-vcodec", "libx264", "-pix_fmt", "yuv420p",
                    "-loglevel", "error", out_path], check=False)
    final_video = out_path if os.path.exists(out_path) and os.path.getsize(out_path) > 0 else raw_path

    # ---- report ----
    duration = fi / fps
    real_ids = [p for p, (a, b) in seen.items() if (b - a) >= 0.5]
    peak = max(counts) if counts else 0
    avg = float(np.mean(counts)) if counts else 0
    mode_count = Counter(counts).most_common(1)[0][0] if counts else 0

    lines = ["## 🤖 Robot report", "",
             f"- **Video length:** {fmt_t(duration)} ({fi} frames)",
             f"- **People in the room at most:** **{peak}**",
             f"- **Typical number of people:** {mode_count} (average {avg:.1f})",
             f"- **Distinct people tracked:** {len(real_ids)} "
             f"*(if people leave/return or get hidden, the tracker may give them a new number)*",
             "", "### What each person did"]
    for pid in sorted(real_ids):
        a, b = seen[pid]
        acts = ", ".join(f"{name} {sec:.0f}s" for name, sec in prim_time[pid].most_common())
        arms = ", ".join(f"{name} {sec:.0f}s" for name, sec in arm_time[pid].most_common())
        lines.append(f"- **Person #{pid}** (seen {fmt_t(a)}–{fmt_t(b)}): {acts}" + (f" · also: {arms}" if arms else ""))

    total_act = Counter()
    for pid in real_ids:
        total_act.update(prim_time[pid])
    if total_act:
        lines += ["", "### Overall room activity",
                  ", ".join(f"{k} – {v:.0f} person-seconds" for k, v in total_act.most_common())]

    step = max(1, len(log) // 60)
    lines += ["", "### Timeline (every second)"] + [f"- {l}" for l in log[::step]]
    return final_video, "\n".join(lines)


with gr.Blocks(title="AI Room Robot") as demo:
    gr.Markdown("# 🤖 AI Room Robot\nUpload a video of a room. The robot counts the people and describes what each one is doing.")
    with gr.Row():
        inp = gr.Video(label="Upload your video")
        outv = gr.Video(label="Robot's view")
    btn = gr.Button("Analyze 🔍", variant="primary")
    rep = gr.Markdown()
    btn.click(analyze, inputs=inp, outputs=[outv, rep])

# Launch as a standalone website (public link, opens in its own browser tab)
from IPython.display import display, HTML
_, local_url, share_url = demo.launch(share=True, inline=False, prevent_thread_lock=True, quiet=True)
display(HTML(f"""
<div style="padding:18px;border-radius:12px;background:#1e1e2e;color:white;font-family:sans-serif">
  <h2 style="margin:0 0 8px 0">🤖 AI Room Robot is ready</h2>
  <a href="{share_url}" target="_blank"
     style="display:inline-block;padding:12px 22px;background:#22c55e;color:white;
            border-radius:8px;text-decoration:none;font-size:18px;font-weight:bold">
     ▶ Open the app in a new window
  </a>
  <p style="margin:10px 0 0 0;opacity:.8">Link: {share_url}<br>
  Keep this Colab cell running while you use the website.</p>
</div>"""))
demo.block_thread()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
* Running on public URL: https://b1f85e542ea0ea9867.gradio.live


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 185ms
Prepared 1 package in 35ms
Installed 1 package in 2ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

